# Black–Litterman Model — Views and Confidence

This notebook defines **investor views** in the Black–Litterman framework
and encodes them in matrix form.

Black–Litterman requires views to be expressed as linear relationships
between expected returns, together with an explicit measure of confidence.
These are represented by:

- **P**: the view matrix  
- **Q**: the view vector  
- **Ω**: the view uncertainty (confidence) matrix  

This notebook focuses **only** on constructing these objects.
No Bayesian update or portfolio optimization is performed here.


## Views specification (conceptual)

We define two simple example investor views.

### 1. Absolute view on AAPL
- We assume AAPL has an expected **excess return of about 8% per year**.
- In Black–Litterman form:  
$ \mu_{\text{AAPL}} \approx 0.08 $

### 2. Relative view: MSFT vs META
- We assume **MSFT will outperform META by about 2% per year** (in excess return terms).
- In Black–Litterman form:  
$ \mu_{\text{MSFT}} - \mu_{\text{META}} \approx 0.02 $

These views are encoded in matrix form using:
- $P$: the view matrix (how each view loads on each asset),
- $Q$: the view vector (the target value of each view),
- $\Omega$: the view uncertainty (confidence) matrix.

In the next step, these objects will be combined with the equilibrium prior
to compute Black–Litterman posterior expected returns.



In [1]:
import numpy as np
import pandas as pd

# Must match the order used in 01_Equilibrium_Prior.ipynb
tickers = ["AAPL", "MSFT", "AMZN", "GOOGL", "META"]
n_assets = len(tickers)

tickers, n_assets


P = np.array([
    [1, 0, 0, 0, 0],    # View 1: AAPL absolute view
    [0, 1, 0, 0, -1]    # View 2: MSFT relative to META
], dtype=float)

P.shape, P


Q = np.array([
    0.08,   # View 1
    0.02    # View 2
], dtype=float).reshape(-1, 1)

Q.shape, Q

view_std_abs = 0.02   # 2% std dev for the AAPL absolute view
view_std_rel = 0.01   # 1% std dev for the MSFT - META spread

Omega = np.diag([
    view_std_abs**2,
    view_std_rel**2
])

Omega.shape, Omega


((2, 2),
 array([[0.0004, 0.    ],
        [0.    , 0.0001]]))

## Interpretation of the view uncertainty matrix $Ω$

The output corresponds to the **view uncertainty (confidence) matrix** $Ω$
used in the Black–Litterman model.

$Ω =
\begin{pmatrix}
0.0004 & 0 \\
0 & 0.0001
\end{pmatrix}$

### Key points

- The matrix has shape $(2, 2)$, matching the number of defined views.
- Each **diagonal entry** represents the variance (uncertainty) of a single view:
  - $0.0004 = 0.02^2$ → uncertainty of the **absolute AAPL return view**
  - $0.0001 = 0.01^2$ → uncertainty of the **MSFT − META relative view**
- Smaller variance implies **higher confidence** in that view.
  - The model assigns higher confidence to the relative outperformance view
    than to the absolute return level.

### Structural interpretation

- Off-diagonal entries are zero, meaning the views are assumed to be
  **independent**.
- This is the standard assumption in baseline Black–Litterman implementations
  and keeps the update transparent and interpretable.

### Role in Black–Litterman

- $Ω$ controls how strongly each view influences the posterior expected returns.
- High uncertainty (large variance) keeps the posterior close to the
  equilibrium prior.
- Low uncertainty (small variance) allows a view to exert greater influence.

This matrix does **not** state that the views are correct.  
It specifies **how much the model is allowed to trust them** relative to the prior.


## Summary of P, Q, and Ω

- **P (view matrix)**  
  Each row corresponds to one view.  
  Each column corresponds to an asset (AAPL, MSFT, AMZN, GOOGL, META).  
  For example:
  - View 1 row: `[1, 0, 0, 0, 0]` → absolute view on AAPL  
  - View 2 row: `[0, 1, 0, 0, -1]` → relative view on MSFT − META  

- **Q (view vector)**  
  Contains the target values for each view (annualized excess returns):  
  - View 1: AAPL ≈ 8%  
  - View 2: MSFT − META ≈ 2%

- **Ω (Omega: view uncertainty matrix)**  
  Diagonal matrix with the variance of each view:  
  - Larger variance → lower confidence  
  - Smaller variance → higher confidence  

These objects \((P, Q, \Omega)\) are the inputs that will be combined with
the equilibrium prior \( \pi \) and the covariance matrix \( \Sigma \)
in the next notebook to compute the **Black–Litterman posterior expected
returns**.
